# 1. Objective
  The Objective of the notebook is to perform data quality checks beforehand and ensure all the basic data requirements are met before proceeding with the later tasks. If there are any significant data issues identified, then the notebook returns an error message
  Following are the data checks that are performed
   - Check if all the required columns as mentioned in the config are present in the harmonized data
   - Check if the data types of the columns are as per expectation
   - Check if Promo related columns are within a range (0-100)
   - Check for data drift and report

# 2. Imports

In [ ]:
# Import python packages

import pandas as pd
import numpy as np
import os

# MUST be set before any numba / evidently import
os.environ["NUMBA_DISABLE_JIT"] = "1"
os.environ["NUMBA_CACHE_DIR"] = "/tmp"

from evidently.report import Report
from evidently.metric_preset import DataDriftPreset
from evidently.options import DataDriftOptions

# df = pd.read_excel('Harmonized Data - Validation.xlsx',sheet_name = 'Sample V2')
# print(df.shape)
# df.head()
# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
print("✅ Snowpark Session Initialized Successfully!")
print("Current Database:", session.get_current_database())
print("Current Schema:", session.get_current_schema())
     

# 3. Setup environment

## 3.1. Load Config

In [ ]:
import yaml
with open("config_new_PROD.yaml") as file:
    app_config = yaml.safe_load(file)

## 3.2. Update Output Database, Schema , table

In [ ]:
output_database = app_config["general_inputs"]["output_database"]
output_schema = app_config["general_inputs"]["output_schema"]
print(output_database, output_schema)

In [ ]:
session.use_database(output_database)
session.use_schema(output_schema)
data_drift_output_table_name = "PROD_DATA_DRIFT_SUMMARY"
data_qc_output_table_name = "PROD_DATA_QC_SUMMARY"

In [ ]:
# Example check (optional)
print("✅ Snowpark Session Initialized Successfully!")
print("Current Database:", session.get_current_database())
print("Current Schema:", session.get_current_schema())

# 4. QC

In [ ]:
df = session.table("PUBLIC.PROD_ADS_STABLE_V4").to_pandas()
print(df.shape)
df.head()

## 4.1. Check if all the required columns are present

In [ ]:
# Checking whether required columns are present in the given data

def check_required_columns(df, required_columns):
    missing = set(required_columns) - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    return True

required_cols = ["F_CODE", "REGIONNAME", "WINBACK_WEIGHTED_MEAN"]
check_required_columns(df, required_cols)

In [ ]:
str(df['REGIONNAME'].dtype)

## 4.2. Check if all the columns are of expected data type

In [ ]:
# checking whether data types are matching.

def validate_dtypes(actual_dtypes: dict, required_dtypes: dict):
    """
    actual_dtypes:  {"col1": "int", "col2": "string"}
    required_dtypes:{"col1": "int", "col2": "float"}
    """
    mismatches = {}

    for col, req_type in required_dtypes.items():
        actual_type = actual_dtypes.get(col)

        if actual_type is None:
            mismatches[col] = f"Missing column (expected {req_type})"
        elif actual_type.lower() != req_type.lower():
            mismatches[col] = f"Expected {req_type}, got {actual_type}"

    return mismatches

actual = {"F_CODE": str(df['F_CODE'].dtype), "REGIONNAME": str(df['REGIONNAME'].dtype)}
required = {"F_CODE": "string", "REGIONNAME": "string"}

errors = validate_dtypes(actual, required)

if errors:
    print("Data type mismatches:")
    for k, v in errors.items():
        print(f"{k}: {v}")
else:
    print("All data types match")

In [ ]:
def split_recent_weeks(df,granularity_cols,week_col,n_weeks):
    
    df = df.copy()

    # rank weeks per granularity (latest = 1)
    df["_week_rank"] = (
        df
        .sort_values(week_col, ascending=False)
        .groupby(granularity_cols)[week_col]
        .rank(method="dense", ascending=False)
    )

    recent_df = df[df["_week_rank"] <= n_weeks]
    history_df = df[df["_week_rank"] > n_weeks]

    return recent_df.drop(columns="_week_rank"), history_df.drop(columns="_week_rank")

recent_df, history_df = split_recent_weeks(df,granularity_cols=["F_CODE","REGIONNAME"],
                                           week_col="START_OF_WEEK",n_weeks=2)

## 4.3. Check if the promo attributes are within the expected range

In [ ]:
def row_level_violations(df):
    cols = [
        c for c in df.columns
        if c.endswith("_WEIGHTED_MEAN") or c.endswith("_NORMALIZED") or c.endswith("_MEAN")
    ]

    violations = (
        df[cols]
        .apply(lambda s: s.where(~s.between(0, 100)))
        .stack()
        .reset_index()
    )

    violations.columns = ["row_index", "column_name", "invalid_value"]
    return violations

df2 = row_level_violations(df)
print(df2)

## 4.4. Data Drift Metrics

In [ ]:
def calculate_psi(expected, actual, bins=10, eps=1e-6):
    """
    expected = reference distribution (historical)
    actual   = current distribution (recent)
    """
    quantiles = np.linspace(0, 1, bins + 1)
    breakpoints = np.unique(np.quantile(expected, quantiles))

    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts = np.histogram(actual, bins=breakpoints)[0]

    expected_pct = expected_counts / len(expected)
    actual_pct = actual_counts / len(actual)

    # avoid divide by zero
    expected_pct = np.where(expected_pct == 0, eps, expected_pct)
    actual_pct = np.where(actual_pct == 0, eps, actual_pct)

    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi

In [ ]:
def psi_by_granularity(history_df,recent_df,granularity_cols,feature_col,bins=10):
    results = []

    for key, hist_grp in history_df.groupby(granularity_cols):
        recent_grp = recent_df[
            (recent_df[granularity_cols] == pd.Series(key, index=granularity_cols)).all(axis=1)
        ]

        if len(hist_grp) == 0 or len(recent_grp) == 0:
            continue

        psi_value = calculate_psi(
            hist_grp[feature_col],
            recent_grp[feature_col],
            bins=bins
        )

        result = dict(zip(granularity_cols, key if isinstance(key, tuple) else [key]))
        result["FEATURE"] = feature_col
        result["PSI"] = psi_value
        results.append(result)

    return pd.DataFrame(results)

psi_df = psi_by_granularity(history_df,recent_df,granularity_cols=["F_CODE","REGIONNAME"],
                            feature_col="NO_OF_NEW_JOINEES",bins=10)

psi_df.head()

In [ ]:
df["START_OF_WEEK"] = pd.to_datetime(df["START_OF_WEEK"])

# Returns historical and recent_4_weeks as dataframe as dictionary
def split_recent_4_weeks_per_fcode(df):
   
    output = {}

    for f_code, g in df.groupby("F_CODE"):
        g = g.sort_values("START_OF_WEEK")

        max_date = g["START_OF_WEEK"].max()
        cutoff_date = max_date - pd.Timedelta(weeks=4)

        current_df = g[g["START_OF_WEEK"] > cutoff_date]
        reference_df = g[g["START_OF_WEEK"] <= cutoff_date]

        if len(current_df) == 0 or len(reference_df) == 0:
            continue  # skip insufficient data

        output[f_code] = {
            "reference": reference_df.drop(columns=["F_CODE", "START_OF_WEEK"]),
            "current": current_df.drop(columns=["F_CODE", "START_OF_WEEK"])
        }

    return output

In [ ]:
# To skip Columns with Insufficient Data

def prepare_for_drift(reference_df, current_df, min_non_null=5):
    valid_cols = [
        col for col in reference_df.columns
        if (
            reference_df[col].notna().sum() >= min_non_null and
            current_df[col].notna().sum() >= min_non_null
        )
    ]
    return reference_df[valid_cols], current_df[valid_cols]


In [ ]:
# Run Evidently Data Drift for Each F_CODE 

fcode_data = split_recent_4_weeks_per_fcode(df)

drift_summary = {}

for f_code, data in fcode_data.items():
    report = Report(metrics=[DataDriftPreset()])

    ref_df, curr_df = prepare_for_drift(data["reference"],data["current"],
                                        min_non_null=5)

    if curr_df.empty or curr_df.shape[1] == 0:
        continue
    
    report.run(
        reference_data=ref_df,
        current_data=curr_df
    )

    drift_summary[f_code] = report.as_dict()

# Extract Column-Level Drift

column_drift_rows = []

for f_code, report_dict in drift_summary.items():
    columns = report_dict["metrics"][0]["result"]["drift_by_columns"]

    for col_name, col_metrics in columns.items():
        column_drift_rows.append({
            "F_CODE": f_code,
            "column": col_name,
            "drift_detected": col_metrics["drift_detected"],
            "drift_score": col_metrics.get("drift_score"),
            "stattest": col_metrics.get("stattest_name"),
            "p_value": col_metrics.get("p_value"),
            "reference_mean": col_metrics.get("reference_statistics", {}).get("mean"),
            "current_mean": col_metrics.get("current_statistics", {}).get("mean")
        })

column_drift_df = pd.DataFrame(column_drift_rows)


column_drift_df


In [ ]:
# drift calculation as an example
reference_df = pd.DataFrame({
    "SALES": [100,110,120,130,140,150],
    "CPI": [1.1,1.1,1.2,1.2,1.3,1.3],
    "DISCOUNT": [5,6,5,6,7,8],
    "REGION": ["N","N","N","N","N","N"]
})

current_df = pd.DataFrame({
    "SALES": [160,170,180,190],
    "CPI": [1.6,1.7,1.8,1.9],
    "DISCOUNT": [9,10,11,12],
    "REGION": ["N","N","N","N"]
})

def prepare_for_drift(reference_df, current_df, min_non_null=2):
    valid_cols = [
        col for col in reference_df.columns
        if (
            reference_df[col].notna().sum() >= min_non_null and
            current_df[col].notna().sum() >= min_non_null
        )
    ]
    return reference_df[valid_cols], current_df[valid_cols]

ref_df, curr_df = prepare_for_drift(reference_df, current_df)

report = Report(
    metrics=[DataDriftPreset()]
    # options=[
    #     DataDriftOptions(enable_embeddings=False)
    # ]
)


report.run(reference_data=ref_df,current_data=curr_df)

report_dict = report.as_dict()

dataset_result = report_dict["metrics"][0]["result"]

dataset_drift_df = pd.DataFrame([{
    "dataset_drift": dataset_result["dataset_drift"],
    "drift_share": dataset_result["drift_share"],
    "n_drifted_columns": dataset_result["number_of_drifted_columns"],
    "n_columns": dataset_result["number_of_columns"]
}])


dataset_drift_df